# AI Heritage Reconstructor - MobileNetV2 training (Google Colab)

Trains a 4-class classifier: `stone_chariot`, `vittala_temple`, `lotus_mahal`, `other`.

**Before you run:** Runtime > Change runtime type > GPU (free T4 is fine).

**What you need on Google Drive:** a folder `heritage_dataset_split/` with `train/`, `val/`, `test/`,
each containing the four class folders. Create it on your PC with `python scripts\split_dataset.py`
and upload the resulting `dataset_split` folder (rename it `heritage_dataset_split`).

**Honesty note:** the numbers this notebook prints are the ONLY accuracy figures you may quote,
and only for the test photos you gave it. Do not quote accuracy before you have run it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, os, numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

DATA_DIR   = '/content/drive/MyDrive/heritage_dataset_split'
OUT_DIR    = '/content/drive/MyDrive/heritage_model_output'
IMG_SIZE   = (224, 224)      # MUST match backend/app/config.py input_size
BATCH      = 32
SEED       = 42
EPOCHS_HEAD, EPOCHS_FINE = 12, 12
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Class order = alphabetical folder order. It is saved to class_names.json and read by the backend.
train_ds = keras.utils.image_dataset_from_directory(f'{DATA_DIR}/train', image_size=IMG_SIZE, batch_size=BATCH, seed=SEED, shuffle=True)
val_ds   = keras.utils.image_dataset_from_directory(f'{DATA_DIR}/val',   image_size=IMG_SIZE, batch_size=BATCH, shuffle=False)
test_ds  = keras.utils.image_dataset_from_directory(f'{DATA_DIR}/test',  image_size=IMG_SIZE, batch_size=BATCH, shuffle=False)
class_names = train_ds.class_names
print('Classes:', class_names)
assert set(class_names) == {'stone_chariot','vittala_temple','lotus_mahal','other'}, 'Unexpected class folders'

# Class weights to compensate for imbalance (counts from the TRAIN split only)
counts = np.zeros(len(class_names))
for _, y in train_ds.unbatch():
    counts[int(y)] += 1
print('Train images per class:', dict(zip(class_names, counts.astype(int))))
total = counts.sum()
class_weight = {i: float(total / (len(class_names) * c)) for i, c in enumerate(counts)}
print('Class weights:', class_weight)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE); val_ds = val_ds.prefetch(AUTOTUNE); test_ds = test_ds.prefetch(AUTOTUNE)

In [ ]:
# Augmentation layers are only active during training and are no-ops at inference.
augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.15),
], name='augmentation')

base = keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,), name='raw_rgb_0_255')   # raw 0-255 pixels
x = augment(inputs)
x = layers.Rescaling(1/127.5, offset=-1, name='mobilenet_preprocess')(x)  # == mobilenet_v2.preprocess_input, INSIDE the model
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation='softmax', name='monument_probs')(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
cb = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)]
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=class_weight, callbacks=cb)

In [ ]:
# Light fine-tuning: unfreeze the top of the backbone, keep BatchNorm layers frozen, use a very small LR.
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE, class_weight=class_weight, callbacks=cb)

In [ ]:
import matplotlib.pyplot as plt
acc  = hist1.history['accuracy']     + hist2.history['accuracy']
vacc = hist1.history['val_accuracy'] + hist2.history['val_accuracy']
loss = hist1.history['loss']         + hist2.history['loss']
vloss= hist1.history['val_loss']     + hist2.history['val_loss']
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(acc, label='train'); ax[0].plot(vacc, label='val'); ax[0].axvline(len(hist1.history['loss'])-1, ls='--', c='gray'); ax[0].set_title('Accuracy'); ax[0].legend()
ax[1].plot(loss, label='train'); ax[1].plot(vloss, label='val'); ax[1].axvline(len(hist1.history['loss'])-1, ls='--', c='gray'); ax[1].set_title('Loss'); ax[1].legend()
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/training_curves.png', dpi=150); plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds)
y_pred = y_prob.argmax(axis=1)
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
print(classification_report(y_true, y_pred, target_names=class_names))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5)); sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='YlOrBr')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix (test set)')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/confusion_matrix.png', dpi=150); plt.show()
test_loss, test_acc = model.evaluate(test_ds)
print(f'TEST accuracy: {test_acc:.4f} on {len(y_true)} test images')

In [ ]:
# Pick a confidence threshold from VALIDATION data (not test), then report its effect on the test set.
val_true = np.concatenate([y.numpy() for _, y in val_ds])
val_prob = model.predict(val_ds)
other_idx = class_names.index('other')
for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    conf = val_prob.max(axis=1); pred = val_prob.argmax(axis=1)
    accepted = (conf >= t) & (pred != other_idx)
    print(f'threshold {t:.1f}: accepted {accepted.mean():.2%} of val images')
print('Choose a value and set HERITAGE_CONFIDENCE_THRESHOLD in the backend (default 0.60).')

In [ ]:
model.save(f'{OUT_DIR}/monument_model.keras')
with open(f'{OUT_DIR}/class_names.json', 'w') as f:
    json.dump(class_names, f)
with open(f'{OUT_DIR}/training_report.json', 'w') as f:
    json.dump({'tensorflow_version': tf.__version__, 'img_size': IMG_SIZE, 'class_names': class_names,
               'train_counts': dict(zip(class_names, counts.astype(int).tolist())),
               'test_accuracy': float(test_acc), 'test_images': int(len(y_true)), 'classification_report': report}, f, indent=2)
print('Saved to', OUT_DIR)

In [ ]:
# Sanity check: reload the saved model and confirm the backend's preprocessing (raw 0-255 pixels) works.
reloaded = keras.models.load_model(f'{OUT_DIR}/monument_model.keras', compile=False)
batch = next(iter(test_ds))[0][:1].numpy()          # raw 0-255 floats, shape (1,224,224,3)
print('Max pixel in batch:', batch.max(), '| probs:', reloaded.predict(batch, verbose=0)[0])

## Next steps
1. Download `monument_model.keras` and `class_names.json` from Drive (`heritage_model_output/`).
2. Copy both into `backend/ai_models/` of the project.
3. Install TensorFlow in the backend venv: `pip install -r requirements-ai.txt`.
4. Restart the backend; `GET /` should show `"model_loaded": true`.
5. Record the printed test accuracy and confusion matrix in your report - **do not** quote numbers you did not measure.